# Recherche d'artistes sur Wikidata

Dépose ton export Spotify à côté de ce notebook, lance les cellules dans l'ordre,
et tu obtiens un fichier de brouillons à relire.

Ce notebook **ne remplit pas la base tout seul**. Wikidata sait dire « condamné
pour X », mais pas si un appel a infirmé, ni quand l'affaire est devenue publique,
ni comment la formuler sans transformer un accusé en coupable. La dernière étape
reste humaine, et c'est elle qui fait la valeur du projet.

Il faut `pandas` et un accès internet. Rien d'autre.


## 1. Réglages

In [ ]:
!pip install pandas

In [ ]:

CSV = "Liked_Songs.csv"        # ton export Exportify, chemin relatif a ce notebook
MIN_TITRES = 5                 # ignorer les artistes en dessous de ce nombre de titres
LIMITE     = 120               # nombre max d'artistes interroges en un passage

BASE    = "../artists.json"          # mettre None si tu n'as pas le depot
REGISTRE = "../review/checked.json"  # idem
SORTIE  = "candidates.json"

# Wikimedia exige un User-Agent descriptif. Mets une adresse ou tu es joignable.
UA = "spotify-feminism-score/0.1 (https://github.com/ClemScb/spotify-feminism-score)"

## 2. Lire la bibliothèque

In [6]:

import json, re, time, unicodedata, urllib.parse, urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd

def normalise(s):
    """Minuscules, sans accents ni ponctuation. Sert a comparer les noms."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

df = pd.read_csv(CSV)

# Exportify separe les artistes multiples par des POINT-VIRGULES.
# Ne jamais couper sur la virgule : cela detruirait "Tyler, The Creator".
col = next(c for c in df.columns if c.lower() in
           ("artist name(s)", "artist name", "artist", "artiste"))

compte = Counter()
for valeur in df[col].dropna():
    for nom in str(valeur).split(";"):
        nom = nom.strip()
        if nom:
            compte[nom] += 1

print(f"{len(df)} titres, {len(compte)} artistes distincts.")
pd.DataFrame(compte.most_common(15), columns=["artiste", "titres"])

2707 titres, 1800 artistes distincts.


,artiste,titres
0,Damso,53
1,Roméo Elvis,50
2,Alpha Wann,48
3,Orelsan,43
4,PLK,41
5,Myd,35
6,SCH,31
7,Justice,29
8,Angèle,28
9,Caballero & JeanJass,27


## 3. Retirer ce qui est déjà connu

On écarte les artistes déjà dans la base et ceux déjà examinés sans résultat,
pour ne pas rechercher éternellement les mêmes.

In [7]:

connus = set()

for chemin, cle in ((BASE, "artists"), (REGISTRE, "checked")):
    if not chemin or not Path(chemin).exists():
        continue
    donnees = json.loads(Path(chemin).read_text(encoding="utf-8"))
    for entree in donnees.get(cle, []):
        nom = entree.get("name", "")
        if nom and not nom.startswith("EXEMPLE"):
            connus.add(normalise(nom))
            for alias in entree.get("aliases", []):
                connus.add(normalise(alias))

a_chercher = [(nom, n) for nom, n in compte.most_common()
              if n >= MIN_TITRES and normalise(nom) not in connus][:LIMITE]

print(f"{len(connus)} noms deja connus, ignores.")
print(f"{len(a_chercher)} artistes a interroger.")
pd.DataFrame(a_chercher, columns=["artiste", "titres"]).head(20)

24 noms deja connus, ignores.
117 artistes a interroger.


,artiste,titres
0,Damso,53
1,PLK,41
2,Myd,35
3,SCH,31
4,Justice,29
5,Angèle,28
6,Caballero & JeanJass,27
7,Fakear,26
8,Casseurs Flowters,26
9,Laura Mvula,24


## 4. Interroger Wikidata

La propriété `P1399` signifie « condamné pour ». On restreint aux humains exerçant
un métier de la musique, sans quoi les homonymes polluent tout.

In [8]:

ENDPOINT = "https://query.wikidata.org/sparql"
LOT = 40

METIERS = ["Q177220", "Q639669", "Q2252262", "Q753110",
           "Q36834", "Q488205", "Q130857", "Q183945"]

GABARIT = """SELECT ?item ?itemLabel ?crime ?crimeLabel ?date ?article WHERE {
  VALUES ?name { %(noms)s }
  VALUES ?occ { %(metiers)s }
  ?item rdfs:label|skos:altLabel ?name .
  ?item wdt:P31 wd:Q5 ; wdt:P106 ?occ .
  OPTIONAL {
    ?item p:P1399 ?st .
    ?st ps:P1399 ?crime .
    OPTIONAL { ?st pq:P585 ?date . }
  }
  OPTIONAL {
    ?article schema:about ?item ; schema:isPartOf <https://fr.wikipedia.org/> .
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "fr,en". }
}"""

def interroger(noms):
    valeurs = " ".join('"%s"@fr "%s"@en' % (n.replace('"', ""), n.replace('"', ""))
                       for n in noms)
    requete = GABARIT % {"noms": valeurs,
                         "metiers": " ".join("wd:" + q for q in METIERS)}
    url = ENDPOINT + "?" + urllib.parse.urlencode({"query": requete, "format": "json"})
    req = urllib.request.Request(url, headers={
        "User-Agent": UA,
        "Accept": "application/sparql-results+json",
    })
    with urllib.request.urlopen(req, timeout=60) as fh:
        return json.load(fh)

noms = [nom for nom, _ in a_chercher]
trouve = {}

for i in range(0, len(noms), LOT):
    lot = noms[i:i + LOT]
    print(f"Lot {i // LOT + 1} / {(len(noms) - 1) // LOT + 1}...")
    reponse = interroger(lot)
    for ligne in reponse["results"]["bindings"]:
        etiquette = ligne["itemLabel"]["value"]
        cle = normalise(etiquette)
        e = trouve.setdefault(cle, {"nom": etiquette,
                                    "wikidata": ligne["item"]["value"],
                                    "wikipedia": ligne.get("article", {}).get("value"),
                                    "chefs": []})
        if "crimeLabel" in ligne:
            chef = {"chef": ligne["crimeLabel"]["value"],
                    "date": ligne.get("date", {}).get("value", "")[:7]}
            if chef not in e["chefs"]:
                e["chefs"].append(chef)
    time.sleep(1.2)   # on reste poli avec un service gratuit

print(f"\n{len(trouve)} artistes identifies sur Wikidata.")

Lot 1 / 3...
Lot 2 / 3...
Lot 3 / 3...

73 artistes identifies sur Wikidata.


## 5. Ce qui ressort

In [9]:

lignes = []
for nom, n in a_chercher:
    hit = trouve.get(normalise(nom))
    lignes.append({
        "artiste": nom,
        "titres": n,
        "sur wikidata": "oui" if hit else "non",
        "chefs declares": ", ".join(c["chef"] for c in hit["chefs"]) if hit else "",
    })

resultats = pd.DataFrame(lignes)
avec_chefs = resultats[resultats["chefs declares"] != ""]

print(f"{len(avec_chefs)} artiste(s) avec au moins une condamnation declaree.")
print(f"{len(resultats) - len(avec_chefs)} sans rien — a inscrire au registre.")
avec_chefs

0 artiste(s) avec au moins une condamnation declaree.
117 sans rien — a inscrire au registre.


,artiste,titres,sur wikidata,chefs declares


## 6. Écrire les brouillons

Le fichier produit suit le format d'`artists.json`. Chaque entrée sort avec
`A_VERIFIER` et une gravité vide : la fusion dans le dépôt refusera tout ce qui
n'aura pas été corrigé à la main.

In [10]:

brouillons = []
for nom, n in a_chercher:
    hit = trouve.get(normalise(nom))
    if not hit or not hit["chefs"]:
        continue
    sources = [hit["wikidata"]] + ([hit["wikipedia"]] if hit["wikipedia"] else [])
    brouillons.append({
        "name": nom,
        "aliases": [hit["nom"]] if normalise(hit["nom"]) != normalise(nom) else [],
        "status": "A_VERIFIER",
        "gravity": None,
        "categories": [c["chef"] for c in hit["chefs"]],
        "public_since": "",
        "summary": "A REDIGER. Wikidata declare : " + " ; ".join(
            c["chef"] + (f" ({c['date']})" if c["date"] else "") for c in hit["chefs"]
        ) + ". Verifier l'issue definitive (appel, relaxe), la date de revelation "
            "publique, et ajouter au moins une source de presse.",
        "sources": sources,
        "_titres_dans_la_bibliotheque": n,
    })

Path(SORTIE).write_text(
    json.dumps({"candidates": brouillons}, ensure_ascii=False, indent=2),
    encoding="utf-8")

print(f"{len(brouillons)} brouillon(s) ecrit(s) dans {SORTIE}.")

0 brouillon(s) ecrit(s) dans candidates.json.


## 7. Mettre à jour le registre

Les noms interrogés sans résultat sont inscrits ici pour ne plus jamais ressortir.
C'est ce qui empêche de rechercher indéfiniment les mêmes artistes.

In [11]:

import datetime

if REGISTRE:
    chemin = Path(REGISTRE)
    chemin.parent.mkdir(parents=True, exist_ok=True)
    donnees = json.loads(chemin.read_text(encoding="utf-8")) if chemin.exists() else {"checked": []}
    entrees = {normalise(e["name"]): e for e in donnees.get("checked", [])}

    aujourdhui = datetime.date.today().isoformat()
    produits = {normalise(b["name"]) for b in brouillons}
    for nom, _ in a_chercher:
        entrees[normalise(nom)] = {
            "name": nom,
            "date": aujourdhui,
            "resultat": "brouillon produit" if normalise(nom) in produits else "rien trouve",
        }

    donnees["checked"] = sorted(entrees.values(), key=lambda e: e["name"].lower())
    chemin.write_text(json.dumps(donnees, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Registre : {len(donnees['checked'])} nom(s) au total.")
else:
    print("Pas de registre configure.")

Registre : 118 nom(s) au total.


## Et ensuite

1. Ouvre `candidates.json` et relis chaque entrée. Lis les sources, tranche le
   statut, fixe la gravité et la date, rédige un résumé factuel, ajoute une source
   de presse. Supprime les homonymes.
2. Depuis le dépôt : `python3 scripts/enrich.py --apply notebooks/candidates.json`
3. Puis `python3 scripts/score.py data/Liked_Songs.csv`

Les trois crans à ne jamais confondre : une **relaxe** prononcée par une
juridiction met la gravité à 0 ; un **classement sans suite** ne tranche rien et
compte comme une accusation ; une **condamnation** est la seule chose qui permet
d'écrire « condamné ».